# Experiment 08 Analysis: GED-Similarity Correlation vs Embedding Size

This notebook analyzes the results of Experiment 08, which systematically compares three molecular representation methods (Morgan fingerprints, MAP4 fingerprints, HDC) across varying embedding sizes for GED correlation analysis.

**Key Research Questions:**
1. How does embedding size affect the correlation between graph edit distance and embedding similarity?
2. Which representation method achieves better structural discrimination?
3. At what embedding size do we see diminishing returns for GED correlation?
4. How does concentration of measure affect similarity distributions at different dimensionalities?

**Primary Metric:** Absolute Pearson correlation between GED and embedding similarity - higher values indicate better structural discrimination.

In [ ]:
%matplotlib inline
import os
import time
import json
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Latex
from rich.pretty import pprint
from pycomex.utils import is_experiment_archive
from pycomex.functional.experiment import Experiment

# This will be the path to the directory in which the notebook is located.
PATH: str = os.getcwd()
# This will have to be the path to the pycomex "results" directory containing the 
# experiment archives of interest.
RESULTS_PATH: str = os.path.join(PATH, 'results')

## Experiment Selection and Sorting Functions

Define how to filter and group experiments based on their parameters.

In [ ]:
# Filter which experiments will be loaded based on their name and/or parameters.
def select_experiment(experiment_name: str,
                      experiment_metadata: dict,
                      experiment_parameters: dict
                      ) -> bool:
    """Select only Experiment 08 similarity experiments."""
    main_condition = 'ex_08_a' in experiment_parameters.get('__PREFIX__', '')
    return main_condition


# Assign a unique key to the experiment based on its data / parameters etc.
def sort_experiment(experiment: Experiment) -> tuple:
    """Sort experiments by (encoding, embedding_size)."""
    
    # Determine encoding type from experiment name and parameters
    experiment_name = experiment.metadata['name']
    print(f'  Processing: {experiment_name}')
    
    # Check fingerprint type parameter to distinguish Morgan FP from MAP4
    fp_type = experiment.parameters.get('FINGERPRINT_TYPE', '')
    
    if fp_type == 'map4':
        encoding = 'fp_map4'
    elif '_fp' in experiment_name or '__fp' in experiment_name:
        encoding = 'fp'
    elif '_hdc' in experiment_name or '__hdc' in experiment_name:
        encoding = 'hdc'
    else:
        encoding = 'unknown'
    
    # Extract embedding size based on encoding type
    if encoding in ('fp', 'fp_map4'):
        embedding_size = experiment.parameters.get('FINGERPRINT_SIZE', 0)
    elif encoding == 'hdc':
        embedding_size = experiment.parameters.get('EMBEDDING_SIZE', 0)
    else:
        embedding_size = 0
    
    return (encoding, embedding_size)

## Experiment Discovery

Find all archived experiments in the results directory.

In [ ]:
# This list will contain the paths to the individual experiment *namespaces* which in 
# turn contain the actual individual experiment archives.
experiment_namespace_paths: list[str] = [
    path
    for file_name in os.listdir(RESULTS_PATH)
    if os.path.isdir(path := os.path.join(RESULTS_PATH, file_name))
]

# Subsequently, this list will contain the paths to the individual experiment archives
# folders.
experiment_paths: list[str] = []
for namespace_path in experiment_namespace_paths:
    for dirpath, dirnames, filenames in os.walk(namespace_path):
        if is_experiment_archive(dirpath):
            experiment_paths.append(dirpath)
            dirnames.clear()  # Prevents further recursion into subdirectories
        
print(f'Found {len(experiment_paths)} experiment archives in {len(experiment_namespace_paths)} namespaces')
pprint(experiment_paths, max_length=5)

## Experiment Loading

Load only the experiments that match our selection criteria (Experiment 08).

In [ ]:
# This list will be populated with the actual Experiment instances which will 
# be loaded from the experiment archive folders.
experiments: list[Experiment] = []

print('Loading experiments from archives...')
time_start: float = time.time()
for experiment_path in experiment_paths:
    
    experiment_identifier: str = os.path.basename(experiment_path)
    
    experiment_data_path = os.path.join(experiment_path, Experiment.DATA_FILE_NAME)
    if not os.path.exists(experiment_data_path):
        continue
    
    experiment_meta_path = os.path.join(experiment_path, Experiment.METADATA_FILE_NAME)
    if not os.path.exists(experiment_meta_path):
        continue
    
    with open(experiment_meta_path) as file:
        content = file.read()
        experiment_metadata: dict = json.loads(content)
        
    if 'parameters' not in experiment_metadata:
        continue
    
    experiment_parameters: dict = {
        param: info['value']
        for param, info in experiment_metadata['parameters'].items()
        if 'value' in info
    }
    
    # Apply filter to determine whether experiment should be included
    condition: bool = select_experiment(
        experiment_name=experiment_metadata['name'],
        experiment_metadata=experiment_metadata,
        experiment_parameters=experiment_parameters
    )
    
    if condition:
        try:
            print(f'   > included experiment "{experiment_identifier}"')
            experiment: Experiment = Experiment.load(experiment_path)
            experiments.append(experiment)
        except Exception as e:
            print(f'   Warning: Failed to load experiment "{experiment_identifier}" - Exception: {e}')
            
time_end: float = time.time()
duration: float = time_end - time_start
print(f'\nLoaded {len(experiments)} experiments in {duration:.2f} seconds')

In [ ]:
# Inspect example experiment data structure
if experiments:
    example_experiment: Experiment = experiments[0]
    print("Example experiment data structure:")
    print(f"Name: {example_experiment.metadata['name']}")
    print(f"\nParameters (sample):")
    for key in ['SEED', 'GED_NUM_SAMPLES', 'NUM_HOPS', 'EMBEDDING_SIZE', 'FINGERPRINT_SIZE']:
        if key in example_experiment.parameters:
            print(f"  {key}: {example_experiment.parameters[key]}")
    print(f"\nData keys:")
    pprint(list(example_experiment.data.keys()), max_length=15)
    
    # Check for GED analysis data
    print(f"\nGED Analysis keys (if available):")
    ged_keys = [k for k in example_experiment.data.keys() if 'ged' in k.lower()]
    pprint(ged_keys, max_length=20)

## Experiment Sorting

Group experiments by encoding method and embedding size.

In [ ]:
# This will be a dictionary mapping the unique key of the experiment to a list of
# experiments which share that key.
key_experiment_map: dict[tuple, list[Experiment]] = defaultdict(list)

for experiment in experiments:
    key: tuple = sort_experiment(experiment)
    key_experiment_map[key].append(experiment)
    
# Sort by encoding name and then by embedding size
key_experiment_map = dict(sorted(
    key_experiment_map.items(), 
    key=lambda item: (item[0][0], item[0][1])  # (encoding, size)
))
    
print(f"\nGrouped experiments into {len(key_experiment_map)} unique configurations")
print("\nConfigurations found:")
for (encoding, size), exps in key_experiment_map.items():
    print(f"  {encoding:8s} | size={size:5d} | n_experiments={len(exps)}")

## Extract GED Correlation Metrics

Extract per-query Pearson correlation and R² values from each experiment's GED analysis. We use the **absolute value** of the correlation so that higher values indicate better structural discrimination.

In [ ]:
# Collect per-query correlation values for each configuration
metrics_data = {}

for (encoding, emb_size), _experiments in key_experiment_map.items():
    
    # Collect ALL per-query correlation values across all experiments
    # We store the ABSOLUTE value so higher = better
    abs_correlation_values = []
    r2_values = []
    
    for exp in _experiments:
        # Load the ged_correlation_summary.csv from the experiment archive
        ged_summary_path = os.path.join(exp.path, 'ged_correlation_summary.csv')
        
        if os.path.exists(ged_summary_path):
            df = pd.read_csv(ged_summary_path)
            
            # Filter out the AGGREGATE row
            df_queries = df[df['query_id'] != 'AGGREGATE']
            
            # Extract per-query correlation values and take absolute value
            if 'correlation' in df_queries.columns:
                raw_corr = df_queries['correlation'].astype(float).tolist()
                abs_correlation_values.extend([abs(c) for c in raw_corr])
            
            if 'r2' in df_queries.columns:
                r2_values.extend(df_queries['r2'].astype(float).tolist())
        else:
            print(f"  Warning: No ged_correlation_summary.csv found in {exp.path}")
    
    # Store all per-query metrics (absolute correlation)
    metrics_data[(encoding, emb_size)] = {
        'abs_correlation': abs_correlation_values,
        'r2': r2_values,
    }
    
    print(f"{encoding:8s} | size={emb_size:5d} | n_samples={len(abs_correlation_values)}")

print(f"\nExtracted metrics for {len(metrics_data)} configurations")

## Box Plot: Absolute Pearson Correlation vs Embedding Size

Create a box plot showing the distribution of |Pearson correlation| values across all 100 query molecules for each embedding size and encoding method. **Higher values indicate better structural discrimination.**

In [ ]:
%matplotlib inline

# ============================================================================
# CONFIGURATION
# ============================================================================

# Embedding sizes to visualize (should match sweep in _slurm_ex_08.py)
EMBEDDING_SIZES = [8, 32, 128, 512, 2048]

# Figure size (width, height) in inches
FIGSIZE = (8, 4)

# Font size
FONT_SIZE = 11

# Color scheme
COLORS = {
    'fp': '#7C92FF',       # Blue for Morgan fingerprints
    'fp_map4': '#FB607F',  # Pink/red for MAP4 fingerprints
    'hdc': '#69F7AE',      # Green for HDC
}

# Labels for legend
LABELS = {
    'fp': 'Morgan FP (Radius 2)',
    'fp_map4': 'MAP4 (Radius 2)',
    'hdc': 'HDC (Depth 2)',
}

# ============================================================================
# PLOT SETUP
# ============================================================================

plt.style.use('default')
plt.rcParams['font.family'] = 'Roboto Condensed'
plt.rcParams['font.size'] = FONT_SIZE

fig, ax = plt.subplots(figsize=FIGSIZE)

# ============================================================================
# DATA PREPARATION FOR BOX PLOTS
# ============================================================================

encodings = ['fp', 'fp_map4', 'hdc']
n_encodings = len(encodings)
box_width = 0.25  # Width of each box
positions_offset = [-(n_encodings - 1) * box_width / 2 + i * box_width for i in range(n_encodings)]

# For each embedding size, create box plots for each encoding
for size_idx, emb_size in enumerate(EMBEDDING_SIZES):
    
    for enc_idx, encoding in enumerate(encodings):
        key = (encoding, emb_size)
        
        if key in metrics_data and metrics_data[key]['abs_correlation']:
            abs_corr_values = metrics_data[key]['abs_correlation']
            
            # Calculate position for this box
            position = size_idx + positions_offset[enc_idx]
            
            # Create box plot
            bp = ax.boxplot(
                [abs_corr_values],
                positions=[position],
                widths=box_width * 0.85,
                patch_artist=True,
                showfliers=True,
                boxprops=dict(facecolor=COLORS[encoding], alpha=0.95, edgecolor='black', linewidth=1.5),
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(color='black', linewidth=1.5),
                capprops=dict(color='black', linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor='black', markersize=4, alpha=0.5),
            )

# ============================================================================
# AXIS CONFIGURATION
# ============================================================================

# Set x-axis labels to embedding sizes
ax.set_xticks(range(len(EMBEDDING_SIZES)))
ax.set_xticklabels(EMBEDDING_SIZES)
ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)

# Set y-axis label (now using absolute correlation, higher is better)
ax.set_ylabel('|Pearson Correlation w. GED| $\\uparrow$', fontsize=FONT_SIZE + 1)

# Set title
ax.set_title('GED-Similarity Correlation by Embedding Size\n(ZINC250k, 100 query molecules per configuration)',
             fontsize=FONT_SIZE + 2, pad=15)

# Set y-axis limits (correlation is between 0 and 1 for absolute values)
ax.set_ylim(0, 1.05)

# Add grid for readability
ax.grid(True, alpha=0.3, axis='y')

# Create custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS[enc], alpha=0.95, edgecolor='black', label=LABELS[enc])
    for enc in encodings
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=FONT_SIZE)

plt.tight_layout()

# Save figure
fig_dir = os.path.join(PATH, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'ex08_ged_correlation_boxplot.svg')
plt.savefig(fig_path)
print(f"Saved figure to {fig_path}")

plt.show()

print("\nInterpretation:")
print("  - Higher |correlation| = better structural discrimination")
print("  - Good embeddings should show: higher GED -> lower similarity")
print("  - Values closer to 1.0 indicate perfect linear relationship")

## LaTeX Table: Summary Statistics

Generate a LaTeX table with mean and standard deviation of |correlation| values for each configuration.

In [ ]:
# ============================================================================
# GENERATE LATEX TABLE
# ============================================================================

# Build table data
table_rows = []

for emb_size in EMBEDDING_SIZES:
    row = {'Embedding Size': emb_size}
    
    for encoding in ['fp', 'fp_map4', 'hdc']:
        label = encoding.upper()
        key = (encoding, emb_size)
        
        if key in metrics_data and metrics_data[key]['abs_correlation']:
            abs_corr_values = metrics_data[key]['abs_correlation']
            r2_values = metrics_data[key]['r2']
            
            corr_mean = np.mean(abs_corr_values)
            corr_std = np.std(abs_corr_values)
            r2_mean = np.mean(r2_values)
            r2_std = np.std(r2_values)
            n = len(abs_corr_values)
            
            row[f'{label} |Corr.|'] = f'{corr_mean:.3f} $\\pm$ {corr_std:.3f}'
            row[f'{label} R$^2$'] = f'{r2_mean:.3f} $\\pm$ {r2_std:.3f}'
        else:
            row[f'{label} |Corr.|'] = 'N/A'
            row[f'{label} R$^2$'] = 'N/A'
    
    table_rows.append(row)

# Create DataFrame
df_table = pd.DataFrame(table_rows)

# Display as regular table first
print("Summary Statistics (using |correlation|, higher is better):")
print("="*100)
display(df_table)

# Generate LaTeX
latex_table = r"""
\begin{table}[htbp]
\centering
\caption{GED-Similarity Correlation by Embedding Size (ZINC250k, n=100 queries per configuration). Higher $|r|$ indicates better structural discrimination.}
\label{tab:ged_correlation}
\begin{tabular}{rcccccc}
\toprule
\textbf{Emb. Size} & \textbf{FP $|r|$} & \textbf{FP R$^2$} & \textbf{MAP4 $|r|$} & \textbf{MAP4 R$^2$} & \textbf{HDC $|r|$} & \textbf{HDC R$^2$} \\
\midrule
"""

for row in table_rows:
    latex_table += f"{row['Embedding Size']} & {row['FP |Corr.|']} & {row['FP R$^2$']} & {row['FP_MAP4 |Corr.|']} & {row['FP_MAP4 R$^2$']} & {row['HDC |Corr.|']} & {row['HDC R$^2$']} \\\\\n"

latex_table += r"""\bottomrule
\end{tabular}
\end{table}
"""

print("\n" + "="*100)
print("LaTeX Table:")
print("="*100)
print(latex_table)

# Save LaTeX to file
latex_path = os.path.join(PATH, 'figures', 'ex08_ged_correlation_table.tex')
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f"\nSaved LaTeX table to {latex_path}")

## Statistical Comparison: Pairwise Tests

Perform statistical tests to quantify differences between all representation methods at each embedding size.

In [ ]:
from scipy import stats
from itertools import combinations

print("\n" + "="*80)
print("PAIRWISE STATISTICAL COMPARISON (using |correlation|, higher is better)")
print("="*80)

encoding_names = ['fp', 'fp_map4', 'hdc']

for emb_size in EMBEDDING_SIZES:
    print(f"\n\nEmbedding Size: {emb_size}")
    print("-" * 50)
    
    # Print means for all methods
    for encoding in encoding_names:
        corr = metrics_data.get((encoding, emb_size), {}).get('abs_correlation', [])
        if corr:
            print(f"  {LABELS.get(encoding, encoding):25s}: {np.mean(corr):.4f} +/- {np.std(corr):.4f}")
        else:
            print(f"  {LABELS.get(encoding, encoding):25s}: no data")
    
    # Pairwise comparisons
    for enc_a, enc_b in combinations(encoding_names, 2):
        corr_a = metrics_data.get((enc_a, emb_size), {}).get('abs_correlation', [])
        corr_b = metrics_data.get((enc_b, emb_size), {}).get('abs_correlation', [])
        
        if not (corr_a and corr_b):
            continue
        
        if len(corr_a) == len(corr_b):
            t_stat, p_value = stats.ttest_rel(corr_b, corr_a)
            test_type = "paired"
        else:
            t_stat, p_value = stats.ttest_ind(corr_b, corr_a)
            test_type = "independent"
        
        diff = np.mean(corr_b) - np.mean(corr_a)
        better = enc_b if diff > 0 else enc_a
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
        
        print(f"\n  {enc_a} vs {enc_b}: {test_type} t={t_stat:.3f}, p={p_value:.4f} {sig} (better: {better}, diff={diff:+.4f})")

print("\n\nSignificance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print("Note: Higher |correlation| indicates better structural discrimination")

## Line Plot: Mean |Correlation| Trends

Line plot showing mean |correlation| across embedding sizes for easier trend identification.

In [ ]:
%matplotlib inline

# ============================================================================
# LINE PLOT: MEAN |CORRELATION| TRENDS
# ============================================================================

plt.style.use('default')
plt.rcParams['font.family'] = 'Roboto Condensed'
plt.rcParams['font.size'] = FONT_SIZE

fig, ax = plt.subplots(figsize=(8, 5))

for encoding in ['fp', 'fp_map4', 'hdc']:
    means = []
    stds = []
    sizes_available = []
    
    for emb_size in EMBEDDING_SIZES:
        key = (encoding, emb_size)
        if key in metrics_data and metrics_data[key]['abs_correlation']:
            abs_corr_values = metrics_data[key]['abs_correlation']
            means.append(np.mean(abs_corr_values))
            stds.append(np.std(abs_corr_values))
            sizes_available.append(emb_size)
    
    if means:
        means = np.array(means)
        stds = np.array(stds)
        sizes_available = np.array(sizes_available)
        
        # Plot line with error bars
        ax.errorbar(sizes_available, means, yerr=stds, 
                   fmt='o-', label=LABELS[encoding], 
                   color=COLORS[encoding], capsize=5, linewidth=2, markersize=8,
                   markeredgecolor='black', markeredgewidth=1.5)

ax.set_xlabel('Embedding Size', fontsize=FONT_SIZE + 1)
ax.set_ylabel('Mean |Pearson Correlation| $\\uparrow$', fontsize=FONT_SIZE + 1)
ax.set_title('GED-Similarity |Correlation| vs Embedding Size\n(Mean +/- Std across 100 queries)',
             fontsize=FONT_SIZE + 2, pad=15)
ax.legend(fontsize=FONT_SIZE)
ax.grid(True, alpha=0.3)

# Set y-axis limits
ax.set_ylim(0, 1)

# Use log scale for x-axis since embedding sizes span multiple orders of magnitude
ax.set_xscale('log', base=2)
ax.set_xticks(EMBEDDING_SIZES)
ax.set_xticklabels(EMBEDDING_SIZES)

plt.tight_layout()

# Save figure
fig_path = os.path.join(fig_dir, 'ex08_correlation_trend.pdf')
plt.savefig(fig_path, bbox_inches='tight', dpi=300)
print(f"Saved figure to {fig_path}")

plt.show()

print("\nInterpretation:")
print("  - Higher |correlation| = better structural discrimination")
print("  - Error bars show standard deviation across query molecules")
print("  - Trend shows how |correlation| changes with embedding dimensionality")

## Summary

Key findings from this analysis.

In [ ]:
print("="*80)
print("EXPERIMENT 08 SUMMARY: GED-Similarity Correlation Analysis")
print("="*80)

print("\nDataset: ZINC250k")
print(f"Embedding sizes tested: {EMBEDDING_SIZES}")
print("Encoding methods: Morgan FP (Radius 2), MAP4 (Radius 2), HDC (Depth 2)")
print("Queries per configuration: 100")
print("Metric: |Pearson correlation| (higher = better structural discrimination)")

print("\n" + "-"*40)
print("KEY FINDINGS:")
print("-"*40)

# Find best configuration for each method
for encoding in ['fp', 'fp_map4', 'hdc']:
    best_size = None
    best_corr = 0
    
    for emb_size in EMBEDDING_SIZES:
        key = (encoding, emb_size)
        if key in metrics_data and metrics_data[key]['abs_correlation']:
            mean_corr = np.mean(metrics_data[key]['abs_correlation'])
            if mean_corr > best_corr:  # Higher is better for absolute correlation
                best_corr = mean_corr
                best_size = emb_size
    
    if best_size:
        print(f"\n{LABELS[encoding]}:")
        print(f"  Best embedding size: {best_size}")
        print(f"  Best mean |correlation|: {best_corr:.4f}")

print("\n" + "="*80)